# **Final Project - Ensemble Models**

- Bryan Keating
- Python and Math for Machine Learning
- 14 December 2025

## Task 1 - Logistic Regression Model Implementation

In [39]:
# Dependencies and libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [40]:
# Load dataset
df = pd.read_csv("allwine.csv")
df.head()

,Unnamed: 0,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,density,pH,sulphates,alcohol,quality
0,0,-0.743787,0.805266,-1.455948,-0.541531,-0.334525,-0.539436,-0.978159,0.146723,-0.755850,-1.297136,0
1,1,-0.520914,1.798500,-1.455948,-0.047918,0.129345,0.787432,-0.998211,-1.220838,-0.062351,-0.960761,0
2,2,-0.520914,1.136344,-1.251203,-0.259467,0.002835,-0.160331,-0.994200,-0.956148,-0.235726,-0.960761,0
3,3,1.373509,-1.512280,1.410480,-0.541531,-0.355610,0.029222,-0.974148,-1.397297,-0.640267,-0.960761,1
4,4,-0.743787,0.805266,-1.455948,-0.541531,-0.334525,-0.539436,-0.978159,0.146723,-0.755850,-1.297136,0


In [41]:
# Check for missing values
df.isnull().sum()

Unnamed: 0             0
fixed acidity          0
volatile acidity       0
citric acid            0
residual sugar         0
chlorides              0
free sulfur dioxide    0
density                0
pH                     0
sulphates              0
alcohol                0
quality                0
dtype: int64

In [42]:
# Split features and target
inp_df = df.drop("quality", axis=1)
out_df = df["quality"]
out_df.head()

0    0
1    0
2    0
3    1
4    0
Name: quality, dtype: int64

In [43]:
# Scale features (even though they are already nomalized)
scaler = StandardScaler()
inp_df = scaler.fit_transform(inp_df)

In [44]:
# Extracting train and test sets
X_train, X_test, y_train, y_test = train_test_split(inp_df, out_df, test_size=0.2, random_state=42)

In [45]:
# Rename and reformat
X_tr_arr = X_train
X_ts_arr = X_test
y_tr_arr = y_train.to_numpy()
y_ts_arr = y_test.to_numpy()

In [46]:
# View data
print('Input Shape:', X_tr_arr.shape)
print('Output Shape:', y_tr_arr.shape)

Input Shape: (2558, 11)
Output Shape: (2558,)


In [47]:
def weightInitialization(n_features):
    w = np.zeros((1, n_features))
    b = 0
    return w, b

In [48]:
def sigmoid_activation(result):
    final_result = 1 / (1 + np.exp(-result))
    return final_result


In [49]:
def model_optimize(w, b, X, Y):
    m = X.shape[0]
    
    # Prediction
    final_result = sigmoid_activation(np.dot(w, X.T) + b)
    Y_T = Y.T
    cost = (-1/m) * (
        np.sum((Y_T * np.log(final_result)) +
               ((1 - Y_T) * (np.log(1 - final_result))))
    )
    
    # Gradient calculation
    dw = (1/m) * (np.dot(X.T, (final_result - Y.T).T))
    db = (1/m) * (np.sum(final_result - Y.T))
    
    grads = {"dw": dw, "db": db}
    
    return grads, cost


In [50]:
def model_predict(w, b, X, Y, learning_rate, no_iterations):
    costs = []
    for i in range(no_iterations):
        grads, cost = model_optimize(w, b, X, Y)
        
        dw = grads["dw"]
        db = grads["db"]
        
        # Weight update (gradient descent step)
        w = w - (learning_rate * dw.T)
        b = b - (learning_rate * db)
        
        if (i % 100 == 0):
            costs.append(cost)
    
    coeff = {"w": w, "b": b}
    gradient = {"dw": dw, "db": db}
    
    return coeff, gradient, costs


In [51]:
def predict(final_pred, m):
    y_pred = np.zeros((1,m))
    for i in range(final_pred.shape[1]):
        if final_pred[0][i] > 0.5:
            y_pred[0][i] = 1
    return y_pred

In [53]:
# Determine model size
n_features = X_tr_arr.shape[1]
print('Number of features:', n_features)

# Initialize model weights
w, b = weightInitialization(n_features)

# Train the model using gradient descent
coeff, gradient, costs = model_predict(
    w, b,
    X_tr_arr, y_tr_arr,
    learning_rate=0.0001,
    no_iterations=4500
)

# Extract trained weights
w = coeff["w"]
b = coeff["b"]
print('Optimized weights', w)
print('Optimized intercept',b)

# Generate probabilities
final_train_pred = sigmoid_activation(np.dot(w, X_tr_arr.T) + b)
final_test_pred = sigmoid_activation(np.dot(w, X_ts_arr.T) + b)

# Convert proabilities to binary prediction
m_tr = X_tr_arr.shape[0]
m_ts = X_ts_arr.shape[0]
y_tr_pred = predict(final_train_pred, m_tr)
y_ts_pred = predict(final_test_pred, m_ts)

# Evaluate model performance
print('Training Accuracy', accuracy_score(y_tr_pred.T, y_tr_arr))
print('Test Accuracy', accuracy_score(y_ts_pred.T, y_ts_arr))

Number of features: 11
Optimized weights [[ 9.18876626e-03  1.77197099e-02 -6.56797996e-02  3.10443617e-02
  -1.19610572e-03 -2.37243168e-02 -1.43054880e-02 -2.13805933e-03
   9.93343389e-05  4.44673413e-02  8.12137352e-02]]
Optimized intercept 0.018816799910001045
Training Accuracy 0.7044566067240031
Test Accuracy 0.6890625


## Task 2 - Dynamic Ensemble Logistic Regression Model

### LR_middle Model Training

In [ ]:
# Determine model size
n_features = X_tr_arr.shape[1]
print('Number of features:', n_features)

# Initialize LR_middle weights
w_M, b_M = weightInitialization(n_features)

# Train LR_middle using gradient descent
coeff_M, gradient_M, costs_M = model_predict(
    w_M, b_M,
    X_tr_arr, y_tr_arr,
    learning_rate=0.0001,
    no_iterations=4500
)

# Extract trained LR_middle weights
w_M = coeff_M["w"]
b_M = coeff_M["b"]
print('Optimized weights (Middle):', w_M)
print('Optimized intercept (Middle):', b_M)

# Generate LR_middle probabilities
h_M_train = sigmoid_activation(np.dot(w_M, X_tr_arr.T) + b_M)
h_M_test  = sigmoid_activation(np.dot(w_M, X_ts_arr.T) + b_M)

# Convert probabilities to binary prediction (sanity check)
m_tr = X_tr_arr.shape[0]
m_ts = X_ts_arr.shape[0]
y_M_tr_pred = predict(h_M_train, m_tr)
y_M_ts_pred = predict(h_M_test, m_ts)

# Evaluate LR_middle performance
print('LR_middle Training Accuracy:', accuracy_score(y_M_tr_pred.T, y_tr_arr))
print('LR_middle Test Accuracy:', accuracy_score(y_M_ts_pred.T, y_ts_arr))